# 02 — Black-Scholes Analytical Pricing

## Theory

The **Black-Scholes-Merton** formula gives an exact closed-form price for a European call option under the following assumptions:

- The underlying follows geometric Brownian motion with constant drift and volatility
- Continuous trading, no transaction costs, no dividends
- A risk-free asset earns constant rate $r$

The call price is:

$$C = S_0\,\Phi(d_1) - K e^{-rT}\,\Phi(d_2)$$

where $\Phi$ is the standard normal CDF and:

$$d_1 = \frac{\ln(S_0/K) + (r + \frac{1}{2}\sigma^2)T}{\sigma\sqrt{T}}, \qquad d_2 = d_1 - \sigma\sqrt{T}$$

Unlike Monte Carlo, this formula is **exact** (within its modelling assumptions) and evaluates in $O(1)$ time — making it the gold-standard benchmark.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '.')

from src.black_scholes import black_scholes_call

## Base Case

In [ ]:
S0, K, r, sigma, T = 100.0, 105.0, 0.05, 0.2, 1.0
price = black_scholes_call(S0, K, r, sigma, T)
print(f"European call price (S0={S0}, K={K}, r={r}, σ={sigma}, T={T}): {price:.4f}")

## Option Price vs Spot Price

We sweep the initial stock price $S_0$ across a range and observe how the call price responds. The dashed vertical line marks the strike $K=105$.

In [ ]:
S_range = np.linspace(50, 200, 300)
call_prices = [black_scholes_call(S, K, r, sigma, T) for S in S_range]
intrinsic = np.maximum(S_range - K, 0)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(S_range, call_prices, color='steelblue', linewidth=2, label='BS call price')
ax.plot(S_range, intrinsic, color='gray', linestyle='--', linewidth=1.2, label='Intrinsic value')
ax.axvline(K, color='crimson', linestyle=':', linewidth=1.5, label=f'Strike K={K}')
ax.axvline(S0, color='darkorange', linestyle=':', linewidth=1.5, label=f'Current spot S₀={S0}')
ax.fill_between(S_range, intrinsic, call_prices, alpha=0.12, color='steelblue', label='Time value')
ax.set_xlabel('Spot price $S_0$')
ax.set_ylabel('Call option price')
ax.set_title('Black-Scholes Call Price vs Spot (K=105, σ=20%, T=1yr)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Sensitivity to Volatility (Vega)

Higher volatility increases option value because it widens the distribution of $S_T$, increasing the probability of large payoffs.

In [ ]:
sigmas = [0.10, 0.15, 0.20, 0.30, 0.40]

fig, ax = plt.subplots(figsize=(9, 5))
for s in sigmas:
    prices_s = [black_scholes_call(S, K, r, s, T) for S in S_range]
    ax.plot(S_range, prices_s, linewidth=1.8, label=f'σ = {int(s*100)}%')

ax.axvline(K, color='black', linestyle=':', linewidth=1, label=f'Strike K={K}')
ax.set_xlabel('Spot price $S_0$')
ax.set_ylabel('Call option price')
ax.set_title('Black-Scholes Call Price for Different Volatilities')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()